# ClipZard Qwen3-4B Clip-Viral LoRA — Colab Training (no Docker, GPU)
Uses `datasets/qwen3_clip_viral_train_9198_combined.jsonl` (9198 rows: 9000 EN + 198 ID) + `validation_1000.jsonl`. No local download needed — Colab will fetch Qwen3-4B via HF_TOKEN.

**Steps:** 1) set HF_TOKEN, 2) upload datasets or clone repo, 3) run training, 4) download LoRA.


In [ ]:
# 0) Check GPU
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# 1) Install deps (torch cu121 for Colab T4/A100)
!pip install -q transformers datasets peft trl accelerate sentencepiece protobuf huggingface_hub tqdm scikit-learn pyyaml python-dotenv bitsandbytes
# optional: hf_transfer for faster HF download
!pip install -q hf_xet 2>&1 | tail -5

In [ ]:
# 2) Get repo + datasets
# Option A: clone your repo (replace URL if private, add token)
# !git clone https://github.com/your-org/clipzard.git
# %cd clipzard

# Option B: upload datasets folder via Colab Files pane (drag datasets/)
# Then verify:
!ls -lh datasets/qwen3_clip_viral_train_9198_combined.jsonl datasets/qwen3_clip_viral_validation_1000.jsonl 2>&1 | head -20
!wc -l datasets/qwen3_clip_viral_train_9198_combined.jsonl datasets/qwen3_clip_viral_validation_1000.jsonl

In [ ]:
# 3) Set HF_TOKEN (paste yours, or use Colab Secrets)
import os
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN (hf_...): ')
os.environ['HF_HOME'] = '/root/.cache/huggingface'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
print('HF_TOKEN set:', os.environ['HF_TOKEN'][:10] + '...')

In [ ]:
# 4) Train — uses training/configs/lora_qwen3_4b_id.yaml (2 epochs, r32, batch1*8, 2048 seq, adamw_torch for Colab)
!python training/scripts/train.py --config training/configs/lora_qwen3_4b_id.yaml 2>&1 | tee training/full.log
# watch live: tail -f training/full.log in another cell

In [ ]:
# 5) Eval (100 samples)
!python training/scripts/evaluate.py --model training/outputs/qwen3-4b-clip-viral-lora-id --max-samples 100

In [ ]:
# 6) Download LoRA (zip and download via Colab)
!zip -r qwen3-4b-lora-id.zip training/outputs/qwen3-4b-clip-viral-lora-id
from google.colab import files
files.download('qwen3-4b-lora-id.zip')

## Tips
- **Free Colab T4 (16GB):** 4B QLoRA r32 with 4bit fits (use `bitsandbytes` 4bit). If OOM, switch to `Qwen/Qwen3-1.7B` in config.
- **A100 (40GB, Colab Pro):** run as-is, 2 epochs ~40min.
- **Resume:** training saves every 500 steps to `outputs/` — if Colab disconnects, rerun cell 4 and it resumes from last checkpoint.
- **Local export:** after download, place LoRA in `training/outputs/` locally and run `python training/scripts/export_gguf.py --hf outputs/qwen3-4b-clip-viral-lora-id/merged`
